# SNF Payroll Approval: Problem Framing And Data Maturity

**Executive takeaway:** SNF administrator teams approve payroll under time pressure. A useful automated flagger must prioritize shift-level exceptions using schedule, timeclock, role, facility, and premium-pay context rather than broad gross/net thresholds.

In [ ]:
import polars as pl
from common.plots import LetsPlot, aes, geom_bar, ggplot, labs, theme_minimal

from payroll_anomaly_ranking.columns import PayrollCol
from payroll_anomaly_ranking.config import PayrollConfig
from payroll_anomaly_ranking.pipeline import run_pipeline
from payroll_anomaly_ranking.presentation import (
    data_quality_summary,
    synthetic_schema_dictionary,
)
from payroll_anomaly_ranking.validation import validate_payroll

LetsPlot.setup_html()

results = run_pipeline(
    PayrollConfig(employee_count=160, pay_periods=12, review_budgets=(10, 25)),
)
payroll = results.payroll
validation = validate_payroll(payroll)

## Privacy And Governance

All records are synthetic. The data contains no real employees, residents, payroll files, tax data, bank data, HR comments, company data, or live integrations. Synthetic anomaly labels are evaluation-only and are excluded from administrator-safe outputs.

In [3]:
synthetic_schema_dictionary()

field_name,business_meaning,type_or_category,privacy_sensitivity,validation_expectation
str,str,str,str,str
"""record_id""","""Synthetic row identifier""","""identifier""","""Low; synthetic only""","""Present and unique enough for …"
"""employee_id""","""Synthetic employee identifier""","""identifier""","""Low; no real employee identity""","""Required and non-null"""
"""pay_period_index""","""Sequential payroll cycle""","""period""","""Low""","""Required and non-null"""
"""pay_period_start""","""Synthetic cycle start date""","""date""","""Low""","""Start date before end date"""
"""pay_period_end""","""Synthetic cycle end date""","""date""","""Low""","""End date after hire date unles…"
…,…,…,…,…
"""hire_date""","""Synthetic hire date""","""date""","""Medium; lifecycle signal""","""Not after pay period end"""
"""termination_date""","""Synthetic termination date whe…","""date""","""Medium; lifecycle signal""","""Pay after termination is a rev…"
"""is_anomaly""","""Injected synthetic evaluation …","""evaluation label""","""Internal synthetic label""","""Retained for evaluation, not s…"


## SNF Shift-Level Data Maturity

The generator creates facilities, units, roles, shift dates, shift types, schedule hours, worked hours, pay codes, premium pay, timeclock quality fields, approval status, and derived pay-period/facility rollups.

In [4]:
data_quality_summary(payroll, validation.warnings)

check,column,message
str,str,str
"""required_column""","""employee_id""","""Missing required column: emplo…"


In [5]:
facility_volume = (
    payroll.group_by(PayrollCol.FACILITY_ID)
    .agg(
        pl.len().alias("shift_lines"),
        pl.sum(PayrollCol.GROSS_PAY).alias("gross_pay"),
        pl.sum(PayrollCol.OVERTIME_HOURS).alias("overtime_hours"),
    )
    .sort(PayrollCol.FACILITY_ID)
)
facility_volume

check,column,message
str,null,str
"""missing_deduction""",null,"""72 records may require payroll…"
"""negative_net_pay""",null,"""26 records may require payroll…"


In [ ]:
(
    ggplot(facility_volume, aes(PayrollCol.FACILITY_ID, "shift_lines"))
    + geom_bar(stat="identity", fill="#396b6f")
    + labs(
        title="Synthetic SNF shift-line volume by facility",
        x="Facility",
        y="Shift-level payroll lines",
    )
    + theme_minimal()
)

## Approval Exception Taxonomy

Initial implemented case-study scenarios focus on overtime/double-shift staffing pressure and premium pay or shift differential mismatch. Future scenario families are documented for agency/float labor, census/acuity, credential/license mismatch, PBJ category mismatch, meal premiums, lifecycle events, retro/rate corrections, union policy variation, new-client bootstrap, and payroll close adjustments.

In [6]:
payroll.group_by(PayrollCol.ANOMALY_CATEGORY).agg(
    pl.len().alias("records"),
    pl.sum(PayrollCol.IS_ANOMALY).alias("synthetic_anomalies"),
    pl.sum(PayrollCol.ANOMALY_DOLLARS).alias("synthetic_anomaly_dollars"),
).sort(PayrollCol.ANOMALY_CATEGORY)

measure,value
str,i64
"""records""",16504
"""pay_periods""",26
"""employees""",650
"""missing_values""",15632
"""invalid_lifecycle_rows""",0
"""terminated_records_with_pay""",29
"""exception_warning_types""",2


## What This Proves

The synthetic dataset has enough SNF-specific context to support weekly pre-approval triage: facility, unit, role, shift, schedule, timeclock, pay-code, premium, and lifecycle fields are available without exposing real payroll data.